# 01. V-World Aerial Image Download

Download V-World satellite/aerial tiles for a target area and merge into a single GeoTIFF.

**Requirements**
- V-World API key: set `VWORLD_API_KEY` in `.env` file at project root
- `python-dotenv`: loads the API key from `.env`
- Conda environment: `svi_segformer`

**Output**
- `data/raw/vworld_aerial_{area_name}.tif` — merged GeoTIFF (EPSG:4326)

In [ ]:
import math
import os
import requests
import numpy as np
from PIL import Image
from io import BytesIO
import rasterio
from rasterio.transform import from_bounds
from rasterio.crs import CRS
from pathlib import Path

# API 키는 .env 파일에서 읽기
# 프로젝트 루트에 .env 파일 생성: VWORLD_API_KEY=your_key_here
from dotenv import load_dotenv
load_dotenv(Path("../../.env"))

API_KEY = os.environ.get("VWORLD_API_KEY")
if not API_KEY or API_KEY == "YOUR_VWORLD_API_KEY":
    raise RuntimeError("Set a valid VWORLD_API_KEY in ../../.env before running this notebook.")

# ── Config ──────────────────────────────────────────────────────────────────
ZOOM = 19           # 19 ≈ 0.25m/px (최고해상도)
AREA_NAME = "test_area"

# 관심 지역 경계 (WGS84 위경도) — 여의도 일부 예시
BBOX = {
    "min_lon": 126.920,
    "min_lat":  37.520,
    "max_lon": 126.935,
    "max_lat":  37.530,
}

OUTPUT_DIR = Path("../../data/raw")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / f"vworld_aerial_{AREA_NAME}.tif"

TILE_URL = "https://api.vworld.kr/req/wmts/1.0.0/{key}/Satellite/{z}/{y}/{x}.jpeg"
TILE_SIZE = 256

In [ ]:
def lon_to_tile_x(lon: float, zoom: int) -> int:
    return int((lon + 180) / 360 * 2**zoom)

def lat_to_tile_y(lat: float, zoom: int) -> int:
    lat_r = math.radians(lat)
    return int((1 - math.log(math.tan(lat_r) + 1 / math.cos(lat_r)) / math.pi) / 2 * 2**zoom)

def tile_to_lon(x: int, zoom: int) -> float:
    return x / 2**zoom * 360 - 180

def tile_to_lat(y: int, zoom: int) -> float:
    n = math.pi - 2 * math.pi * y / 2**zoom
    return math.degrees(math.atan(math.sinh(n)))

x_min = lon_to_tile_x(BBOX["min_lon"], ZOOM)
x_max = lon_to_tile_x(BBOX["max_lon"], ZOOM)
y_min = lat_to_tile_y(BBOX["max_lat"], ZOOM)  # lat↑ → tile_y↓
y_max = lat_to_tile_y(BBOX["min_lat"], ZOOM)

n_x = x_max - x_min + 1
n_y = y_max - y_min + 1
print(f"Tile grid: {n_x} x {n_y} = {n_x * n_y} tiles")
print(f"Canvas size: {n_x * TILE_SIZE} x {n_y * TILE_SIZE} px")

In [ ]:
canvas = np.zeros((n_y * TILE_SIZE, n_x * TILE_SIZE, 3), dtype=np.uint8)

session = requests.Session()
total = n_x * n_y
downloaded = 0

for row, y in enumerate(range(y_min, y_max + 1)):
    for col, x in enumerate(range(x_min, x_max + 1)):
        url = TILE_URL.format(key=API_KEY, z=ZOOM, y=y, x=x)
        try:
            resp = session.get(url, timeout=10)
            resp.raise_for_status()
            img = np.array(Image.open(BytesIO(resp.content)).convert("RGB"))
            r0, r1 = row * TILE_SIZE, (row + 1) * TILE_SIZE
            c0, c1 = col * TILE_SIZE, (col + 1) * TILE_SIZE
            canvas[r0:r1, c0:c1] = img
        except Exception as e:
            print(f"  [WARN] tile ({x},{y}) failed: {e}")
        downloaded += 1
        if downloaded % 10 == 0 or downloaded == total:
            print(f"  {downloaded}/{total} tiles done", end="\r")

print(f"\nDone. Canvas: {canvas.shape}")

In [ ]:
west  = tile_to_lon(x_min, ZOOM)
east  = tile_to_lon(x_max + 1, ZOOM)
north = tile_to_lat(y_min, ZOOM)
south = tile_to_lat(y_max + 1, ZOOM)

transform = from_bounds(west, south, east, north, canvas.shape[1], canvas.shape[0])

with rasterio.open(
    OUTPUT_PATH, "w",
    driver="GTiff",
    height=canvas.shape[0],
    width=canvas.shape[1],
    count=3,
    dtype=np.uint8,
    crs=CRS.from_epsg(4326),
    transform=transform,
    compress="lzw",
) as dst:
    for i in range(3):
        dst.write(canvas[:, :, i], i + 1)

print(f"Saved: {OUTPUT_PATH}")
print(f"Bounds: W={west:.6f}, S={south:.6f}, E={east:.6f}, N={north:.6f}")

In [ ]:
import matplotlib.pyplot as plt

with rasterio.open(OUTPUT_PATH) as src:
    preview = np.stack([src.read(1), src.read(2), src.read(3)], axis=-1)

fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(preview)
ax.set_title(f"V-World Aerial — {AREA_NAME} (zoom={ZOOM})")
ax.axis("off")
plt.tight_layout()
plt.show()